In [4]:
# Install required packages
!pip install transformers torch accelerate datasets

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score, classification_report
import os

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
GPU: Tesla T4


In [6]:
# LOAD DATA FROM GOOGLE DRIVE

DRIVE_PATH = "/content/drive/MyDrive/bible_classifier_data/"

# Load your text data - you'll need to create these files
# Option 1: If you have the raw text data
try:
    # Load from CSV files (you need to create these from your original data)
    train_df = pd.read_csv(f"{DRIVE_PATH}train_texts.csv")
    test_df = pd.read_csv(f"{DRIVE_PATH}test_texts.csv")

    # Assuming columns: 'text', 'label' (0 or 1)
    X_train_texts = train_df['text'].tolist()
    y_train = train_df['label'].tolist()
    X_test_texts = test_df['text'].tolist()
    y_test = test_df['label'].tolist()

except FileNotFoundError:
    print("CSV files not found. Please create train_texts.csv and test_texts.csv")
    print("with columns: 'text' and 'label' (0=Not Bible, 1=Bible)")
    exit()

print(f"Training samples: {len(X_train_texts)}")
print(f"Test samples: {len(X_test_texts)}")
print(f"Label distribution - Train: {pd.Series(y_train).value_counts().to_dict()}")

Training samples: 49976
Test samples: 12495
Label distribution - Train: {0: 25094, 1: 24882}


In [7]:
# DATASET CLASS

class BibleTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [8]:
# INITIALIZE MODEL AND TOKENIZER

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Create datasets
train_dataset = BibleTextDataset(X_train_texts, y_train, tokenizer)
test_dataset = BibleTextDataset(X_test_texts, y_test, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# TRAINING SETUP

# Training arguments optimized for T4 GPU
training_args = TrainingArguments(
    output_dir=f'{DRIVE_PATH}roberta_bible_results',
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Adjust based on GPU memory
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f'{DRIVE_PATH}roberta_bible_logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,  # Use mixed precision for T4
    dataloader_pin_memory=False,
    report_to="none"  # Disable wandb logging
)

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions)
    }

In [ ]:
# TRAINER AND TRAINING

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

In [ ]:
# EVALUATION

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(test_dataset)
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

# Get detailed predictions
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Bible', 'Bible']))

In [ ]:
# ===============================================
# SAVE MODEL
# ===============================================

# Save the fine-tuned model
model.save_pretrained(f'{DRIVE_PATH}roberta_bible_model')
tokenizer.save_pretrained(f'{DRIVE_PATH}roberta_bible_model')

In [10]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import torch

# Paths
MODEL_PATH = f"{DRIVE_PATH}roberta_bible_model"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Reload test data
test_df = pd.read_csv(f"{DRIVE_PATH}test_texts.csv")
X_test_texts = [str(t) for t in test_df['text'].tolist()]
y_test = test_df['label'].tolist()

# Create dataset
test_dataset = BibleTextDataset(X_test_texts, y_test, tokenizer)

# Create Trainer with logging/reporting disabled
args = TrainingArguments(
    output_dir="/tmp",
    per_device_eval_batch_size=16,
    report_to="none",       # disables wandb, comet, etc.
    do_train=False,
    do_eval=False
)

trainer = Trainer(model=model, args=args)

# Predict
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, target_names=['Not Bible', 'Bible']))


Test Accuracy: 0.9817
              precision    recall  f1-score   support

   Not Bible       0.98      0.99      0.98      6274
       Bible       0.99      0.98      0.98      6221

    accuracy                           0.98     12495
   macro avg       0.98      0.98      0.98     12495
weighted avg       0.98      0.98      0.98     12495



In [11]:
# ===============================================
# INTERACTIVE INFERENCE WITH PIPELINE
# ===============================================

from transformers import pipeline
import torch

MODEL_PATH = f"{DRIVE_PATH}roberta_bible_model"

# Create pipeline
classifier = pipeline(
    "text-classification",
    model=MODEL_PATH,
    tokenizer=MODEL_PATH,
    device=0 if torch.cuda.is_available() else -1
)

# Label mapping
label_map = {"LABEL_0": "Not Bible", "LABEL_1": "Bible"}

print("\n" + "="*50)
print("RoBERTa Bible Classifier - Interactive Mode")
print("Type text to classify (or 'quit' to exit)")
print("="*50)

while True:
    user_input = str(input("\n>>> "))
    if user_input.lower() in ["quit", "exit", "stop"]:
        break

    if user_input.strip():
        result = classifier(user_input, truncation=True)
        pred_label = label_map[result[0]["label"]]
        confidence = result[0]["score"]
        print(f"Prediction: {pred_label} (confidence {confidence:.4f})")


Device set to use cuda:0



RoBERTa Bible Classifier - Interactive Mode
Type text to classify (or 'quit' to exit)

>>> in the beginning, god created the heavens and the earth
Prediction: Bible (confidence 0.9976)

>>> haste is the enemy of perfection
Prediction: Bible (confidence 0.9974)

>>> It is not good to act without thinking; and he who hurries with his feet errs in the way
Prediction: Bible (confidence 0.9976)

>>> Pride goes before destruction.
Prediction: Not Bible (confidence 0.9931)

>>> Not a leaf falls from a tree unless it is God's will.
Prediction: Bible (confidence 0.9976)

>>> Some pretend to be rich and have nothing...
Prediction: Not Bible (confidence 0.9964)

>>> It is in giving that one receives.
Prediction: Not Bible (confidence 0.9742)

>>> An idle mind is the devil's workshop.
Prediction: Not Bible (confidence 0.9988)

>>> If you show yourself weak on the day of trouble, your strength will be small.
Prediction: Bible (confidence 0.9976)

>>> Render unto Caesar the things that are Caesar's

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Prediction: Bible (confidence 0.9976)

>>> Cast your bread upon the waters, for you will find it after many days.
Prediction: Bible (confidence 0.9976)

>>> quit
